In [ ]:
%load_ext autoreload
%autoreload 2

# Libraries

In [ ]:
# Librerías solo las que realmente se usan en ESTE notebook (verificado
# con grep, no por inspección visual). Se quitaron, por no usarse en ningún
# lado: HEALTHY_LABEL_MAPPING, StandardScaler, make_scorer, seaborn,
# GAModelSearchHealthy, MakeDataset, BREAST_DENSITY_LABEL_MAPPING,
# MASTER_LABEL_MAPPING, data_interim_dir, data_raw_dir, reports_dir.
# El mapeo de etiquetas (BI-RADS -> clase, breastDensity -> 0/1) pasa DENTRO
# de pipelines/data/build_training_dataset.py (importado más abajo como
# `btd`), no en este notebook -- por eso esos mapeos no hacían falta acá.

# Básicas: arrays, tablas, gráficos, guardar el modelo ganador
import joblib
import matplotlib.pyplot as plt

# --- Tracking de experimentos: cada clasificador de la comparación final
# queda logueado acá (params, métricas, figuras, modelo) bajo el experimento
# "entrenamiento_healthy_multiclasificador_qc". Ver cómo abrir la UI al
# final del notebook.
import mlflow
import numpy as np
import pandas as pd
from dotenv import dotenv_values
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

# Preprocesamiento y validación cruzada
from sklearn.preprocessing import MinMaxScaler
from sklearn_genetic import GASearchCV

# Búsqueda genética de hiperparámetros -- SOLO se usa en la sección
# opcional de selección L1 de features (ver más abajo), no en la
# comparación de clasificadores final (esa usa Optuna, celda de "Training"). ---
from sklearn_genetic.space import Categorical, Continuous

# Rutas del proyecto (independientes de desde qué carpeta se ejecute).
# data_processed_dir: para leer el CSV de features ya calculado.
# models_dir: para guardar el .pkl del modelo ganador al final.
from julieta.utils.paths import data_processed_dir, models_dir, project_dir

# Unico cambio real vs el notebook original: apuntar mlflow al tracking
# server de Azure ML de este repo -- el original no llamaba set_tracking_uri()
# en ningun lado (dependia de un mlflow local). Ver ADR 0005/0009.
mlflow.set_tracking_uri(dotenv_values(project_dir(".env"))["MLFLOW_TRACKING_URI"])

# Load data

## Diagnóstico y categóricos -- CÓMO SE CARGAN REALMENTE HOY

**Igual que con las desconexiones**: las celdas que había aquí antes (leer `mammography_data_{SURA,CAFAM,CLINICA_DE_MAMA}.csv`, limpiar texto en español, leer y limpiar `categoricals_data_{...}.csv`) ya NO se usan y las borré -- eran código heredado de la plantilla original que quedaba corriendo sin que nada dependiera de su resultado.

**Dónde pasa la carga real**: dentro de `pipelines/data/build_training_dataset.py`:
- `_load_diagnosis(studies)` -- lee y limpia `mammography_data_<estudio>.csv`, descarta `BI RADS 0`.
- `_load_categoricals(studies, df_diagnosis)` -- lee y limpia `categoricals_data_<estudio>.csv`. Incluye el fix de `height`/`weight` == 0 tratado como NaN (antes corrompía el promedio combinado: la mediana de altura en SURA era literalmente 0.0).

La celda de "Load features" más abajo llama a estas dos funciones directamente para armar `X`, así que lo que ves ahí SÍ es lo que realmente entrena el modelo.

## Desconexiones -- CÓMO SE FILTRAN REALMENTE HOY

**Las celdas que había aquí antes (leer `disconnections.xlsx` + `disconnection_maicao.csv`) ya NO se usan y las borré** -- eran código heredado de la plantilla original y causaban confusión real: alguien podía mirarlas y pensar que solo se filtra con esos dos archivos, cuando el filtrado de verdad pasa en otro lado.

**Dónde pasa el filtrado de verdad**: dentro de `pipelines/data/build_training_dataset.py`, función `_load_disconnections(studies)`. Combina:
1. `disconnections.xlsx` + `disconnection_maicao.csv` (el reetiquetado manual original).
2. **Un archivo de control de calidad POR ESTUDIO** (`disconnection_sura.csv`, `disconnection_cafam.csv`, `disconnection_cmtest.csv` = CLINICA_DE_MAMA, `disconnection_santafe.csv`) -- esto es lo que faltaba antes: esos 4 archivos existían pero nadie los usaba. Cada uno tiene un esquema distinto (SANTAFE/CAFAM tienen columnas `left`/`right`; CAFAM/CLINICA_DE_MAMA tienen `valid`; SURA solo tiene `gaps_detected`/`needs_interpolation`), así que se excluye por la señal "fuerte" que exista en cada archivo (nunca por `gaps_detected` solo, que es demasiado ruidoso).

Con este fix, se excluyeron **114 pacientes adicionales** de SURA/CAFAM/CLINICA_DE_MAMA que antes se colaban al entrenamiento sin filtrar. El dataset cacheado que se lee más abajo (`healthy_advanced_nyquist_dataset_qc.csv`) ya tiene este filtro aplicado.

# Load features

In [ ]:
# FLUJO DE FEATURES -- EXPERIMENTO magnitud/fase (copia de este notebook,
# ver VMV-healthy_model_experimentos.ipynb para la versión con
# resistencia/reactancia). Dataset ya corregido por control de calidad
# (114 pacientes de SURA/CAFAM/CLINICA_DE_MAMA excluidos por sus propios
# archivos de desconexión) + categóricas frescas con el fix de
# height/weight==0 -> NaN.
#
# feature_mode="advanced_magphase" (ComputeAdvancedMagPhaseFeatures) usa
# EXACTAMENTE el mismo esquema de 59 features/nodo que "advanced_nyquist",
# pero calculado sobre la curva (magnitud, fase) en vez de (resistencia,
# reactancia): geometría del arco sobre |Z|/fase, ajustes Cole-Cole/RS-CPE
# con el error medido en log|Z|+fase. El bloque Bode (12 columnas) sale
# idéntico entre ambos modos, porque ya estaba basado en magnitud/fase desde
# el origen -- el cambio real está en los 34 features geométricos y los 13
# paramétricos.

import sys

from julieta.utils.paths import project_dir

sys.path.insert(0, str(project_dir("pipelines", "data")))
import build_training_dataset as btd  # noqa: E402

CATEGORICAL_COLUMNS = [
    "age",
    "braCupSize",
    "height",
    "weight",
    "hormonalContraception",
    "hormonalTherapyTreatment",
    "menopause",
]
LABELED_STUDIES = ["SURA", "CAFAM", "CLINICA_DE_MAMA"]

df_completo = pd.read_csv(
    data_processed_dir("healthy_advanced_magphase_dataset_qc.csv"), index_col=[0, 1]
)
context_cols = [
    c
    for c in ["mammogramCategory", "label", "study", "relevantFindingLaterality", "breastDensity"]
    if c in df_completo.columns
]
X_signal = df_completo.drop(columns=context_cols)
y = df_completo["label"]
data_study = df_completo["study"]

df_diagnosis_fresh = btd._load_diagnosis(LABELED_STUDIES)
df_categoricals_fresh = btd._load_categoricals(LABELED_STUDIES, df_diagnosis_fresh)[
    CATEGORICAL_COLUMNS
]

patient_ids = X_signal.index.get_level_values("patient_id")
categoricals_aligned = df_categoricals_fresh.reindex(patient_ids)
categoricals_aligned.index = X_signal.index

X = pd.concat([X_signal, categoricals_aligned], axis=1).dropna()
y = y.reindex(X.index)
data_study = data_study.reindex(X.index)

print(f"X shape: {X.shape}")
print("Label distribution")
print(y.value_counts().sort_index())

# Data splitting and processing

`DataSplitter` (`src/julieta/data/split_data.py`) hace un split 70/30 estratificado por `y` (mismo `random_state=1124` fijo siempre, así que este split es idéntico entre corridas -- no cambia aunque cambie el resto del código). Estratificado significa que la proporción de clases 0/1/2 se mantiene igual en train_val y en test, no se reparte al azar.

`X_test`/`y_test` no se vuelven a tocar hasta la evaluación final de cada clasificador -- todo lo que pasa en las secciones de "Feature selection" y "Training" (incluyendo la búsqueda de hiperparámetros de Optuna) usa únicamente `X_train_val`/`y_train_val`, para que la métrica de test sea una estimación honesta y no optimista.

In [ ]:
from julieta.data.split_data import DataSplitter

splitter = DataSplitter()

# split_by_group (no split()): agrupa por patient_id antes de partir, para que
# ambos senos del mismo paciente queden siempre del mismo lado (train o test).
# Se confirmo empiricamente (notebook combinado rx+mp) que el mismo paciente es
# 45% mas parecido entre sus dos senos que dos pacientes distintos de la misma
# clase (p=2e-204) -- partir por fila sin agrupar deja pasar esa correlacion de
# sesion de medicion al split, inflando el J de Youden en test.
X_train_val, X_test, y_train_val, y_test = splitter.split_by_group(
    X, y, group_level="patient_id", test_size=0.25
)
y_train_val = y_train_val
y_test = y_test

train_patients = set(X_train_val.index.get_level_values("patient_id"))
test_patients = set(X_test.index.get_level_values("patient_id"))
assert not (train_patients & test_patients), "Fuga: hay pacientes en ambos lados del split."
print(
    f"Pacientes en train_val: {len(train_patients)} | en test: {len(test_patients)} | overlap: {len(train_patients & test_patients)}"
)

In [ ]:
print("Label distribution - train_val data set")
print(y_train_val.value_counts().sort_index())

print("Label distribution - test data set")
print(y_test.value_counts().sort_index())

### Chequeo visual rápido (PCA)

Esto es solo exploración, no una decisión de modelado: se comprimen las 184 columnas a 2 dimensiones con PCA (Análisis de Componentes Principales, la forma más simple de "aplanar" muchas columnas a un plano 2D conservando la mayor variabilidad posible) y se grafican los pacientes de `X_train_val` dos veces:

1. Coloreados por `label` (0/1/2) -- para ver a simple vista si las clases se separan algo en ese plano, o si están todas mezcladas (lo segundo es normal y esperado: si se separaran limpiamente en 2D, el problema sería trivial y no haría falta nada de lo que sigue).
2. Coloreados por `study` (SURA/CAFAM/CLINICA_DE_MAMA) -- para ver si hay un efecto de "estudio" muy fuerte en las features (que las mediciones de una clínica se agrupen aparte de las otras por razones que no tienen que ver con el diagnóstico, ej. calibración del equipo). Si los 3 estudios se ven mezclados entre sí, es buena señal de que el modelo no está aprendiendo a distinguir clínicas en vez de pacientes.

In [ ]:
from julieta.visualization.projections import VisualizeProjections

%matplotlib inline
visualizer = VisualizeProjections()
visualizer.visualize_projection(X_train_val, MinMaxScaler(), method="PCA", y=y_train_val)
visualizer.visualize_projection(
    X_train_val, MinMaxScaler(), method="PCA", y=data_study.loc[y_train_val.index]
)

# Feature selection

Feature selection basada en modelos lineales: se usan los coeficientes de una LogisticRegression con `penalty='l1'` (Lasso), que tiende a dejar coeficientes en cero para variables irrelevantes -- una forma clásica de filtrar columnas.

**Por qué esto era necesario en el modo `raw` (~2880 columnas) y por qué hoy casi nunca se ejecuta**: con 2880 columnas y ~1000 filas de entrenamiento se cae en el régimen `p >> n` (más columnas que filas), donde casi cualquier modelo sobreajusta sin algún tipo de reducción antes de entrenar. Con `feature_mode="advanced_nyquist"` (184 columnas, ver "Load features" arriba) ya no estamos en ese régimen, así que la celda de abajo se salta esta selección automáticamente (columnas ≤ `SKIP_L1_SELECTION_THRESHOLD=300`) y usa las 184 columnas completas. El código de selección genética se deja funcionando (por si en el futuro se agregan más señales/estudios y las columnas vuelven a subir de 300), pero **hoy no se ejecuta con este dataset** -- lo vas a ver confirmado en el print de la celda siguiente.

In [ ]:
# Con ~2880 columnas crudas hacía falta L1 para no entrenar con p >> n.
# Con feature_mode="advanced_nyquist" (~177 columnas físicas) ya no estamos
# en ese régimen, así que esta selección se vuelve OPCIONAL: se salta si ya
# hay pocas columnas relativas al tamaño de la muestra, y solo se activa si
# aún hace falta reducir más.
SKIP_L1_SELECTION_THRESHOLD = 300  # columnas; por debajo de esto no seleccionamos

if X_train_val.shape[1] <= SKIP_L1_SELECTION_THRESHOLD:
    print(
        f"X_train_val tiene {X_train_val.shape[1]} columnas (<= {SKIP_L1_SELECTION_THRESHOLD}); "
        "se omite la selección L1, se usan todas las columnas."
    )
    selected_features = X_train_val.columns.tolist()
    ga_search_lr = None
else:
    # 1. Pipeline base
    pipe = Pipeline(
        [
            ("scaler", MinMaxScaler()),
            (
                "logreg",
                LogisticRegression(
                    class_weight="balanced",
                    solver="liblinear",  # necesario para L1
                    max_iter=2000,
                ),
            ),
        ]
    )

    # 2. Espacio de hiperparámetros
    param_grid = {
        "logreg__penalty": Categorical(["l1"]),
        "logreg__C": Continuous(0.001, 10.0, distribution="log-uniform"),
    }

    # 3. Optimización genética
    ga_search_lr = GASearchCV(
        estimator=pipe,
        cv=StratifiedKFold(n_splits=5),
        scoring="f1_macro",
        param_grid=param_grid,
        population_size=2,
        generations=2,
        verbose=True,
        n_jobs=-1,
    )

    ga_search_lr.fit(X_train_val, y_train_val)

    print(" Mejores hiperparámetros encontrados:\n", ga_search_lr.best_params_)
    print(" Mejor f1-score macro en cross-val:", ga_search_lr.best_score_)

    best_lr = ga_search_lr.best_estimator_.named_steps["logreg"]

    # Gráfico de evolución del fitness
    history = ga_search_lr.history
    generations = history["gen"]
    fitness_mean = np.array(history["fitness"])
    fitness_std = np.array(history["fitness_std"])
    fitness_max = history["fitness_max"]
    fitness_min = history["fitness_min"]

    plt.figure(figsize=(10, 6))
    plt.plot(generations, fitness_mean, label="Mean Fitness", marker="o")
    plt.fill_between(
        generations,
        fitness_mean - fitness_std,
        fitness_mean + fitness_std,
        alpha=0.2,
        label="Fitness ±1 std",
    )
    plt.plot(generations, fitness_max, label="Max Fitness", linestyle="--")
    plt.plot(generations, fitness_min, label="Min Fitness", linestyle="--")
    plt.axvline(np.argmax(fitness_mean), linestyle="--", label="Optimal Generation")
    plt.xlabel("Generation")
    plt.ylabel("Fitness (F1 macro)")
    plt.title("Fitness Progression Across Generations (Logistic Regression)")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 4. Selección de features (post-entrenamiento)
# Solo aplica si se corrió la selección L1 arriba (columnas > SKIP_L1_SELECTION_THRESHOLD).
if ga_search_lr is not None:
    best_lr = ga_search_lr.best_estimator_.named_steps["logreg"]
    # Relevancia para la clase 2 = maligno
    coefs = np.abs(best_lr.coef_[2])
    feature_importance = pd.Series(coefs, index=X_train_val.columns).sort_values(ascending=False)

    top_k = int(len(feature_importance) * 0.05)
    selected_features = feature_importance.head(top_k).index.tolist()
    print(f"Importancia mas alta {np.max(feature_importance[selected_features])}")
    print(f"Importancia mas baja {np.min(feature_importance[selected_features])}")

    print(f" Se seleccionaron {len(selected_features)} de {len(feature_importance)} features (5%)")
else:
    print(f" Sin selección adicional: se usan las {len(selected_features)} columnas ya existentes.")

In [ ]:
# 5. Reducir los datasets
X_train_val_reduced = X_train_val[selected_features]
X_test_reduced = X_test[selected_features]

# Training

In [ ]:
# Cross-validation strategy -- se mantiene FIJA (misma variable `cv`) para
# TODOS los modelos de la comparación de abajo, así que ningún clasificador
# tiene ventaja por evaluarse con una partición distinta.
# - n_splits=5: 5 particiones -- cada fold entrena con el 80% de
#   X_train_val y valida con el 20% restante, rotando.
# - shuffle=True + random_state=369: se mezclan las filas antes de partir
#   (si no, quedarían agrupadas por orden de carga: SURA, luego CAFAM, luego
#   CLINICA_DE_MAMA) y con semilla fija para que los folds sean siempre los
#   mismos entre corridas.
# - StratifiedKFold (no KFold a secas): cada fold mantiene la misma
#   proporción de clases 0/1/2 que el total -- importante acá porque la
#   clase 2 (maligno) es la minoritaria (~23% de los datos).
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=369)

# triclass=True porque `label` tiene 3 valores (0/1/2), no 2. Esto le dice a
# ClassificationMetrics que, para calcular sensibilidad/especificidad/PPV/NPV,
# primero debe binarizar: clase 2 (maligno/alta sospecha) = "positivo",
# clases 0+1 (normal/intermedio) = "negativo" -- ver map_to_binary() en
# src/julieta/models/metrics.py. Sin esto, esas métricas no tendrían sentido
# para un problema de 3 clases.
triclass = y_train_val.nunique() > 2
print(f"triclass={triclass}")

# Comparación de varios clasificadores, todos optimizados por J de Youden

`GAModelSearch.genopt_training` elige "ganador" por un score agregado (f1_macro sobre el problema binarizado), y ese score puede declarar ganador a un modelo con sensibilidad pésima en la clase maligna -- así pasó con XGBoost (score 0.68, sensibilidad real 0.13-0.19 según el dataset). Para un modelo de priorización de pacientes, la sensibilidad de la clase maligna ES la métrica que importa, no el score agregado.

Acá se comparan **7 configuraciones de clasificador** (SVC, LogisticRegression, RandomForest, XGBoost, XGBoost con balanceo de clases, GradientBoosting, KNN), cada una con su propia búsqueda de hiperparámetros en Optuna optimizando **directamente el índice J de Youden** (sensibilidad + especificidad - 1) vía cross-validation -- nunca sensibilidad sola (ya vimos que eso colapsa a "marcar a todos como malignos").

**Por qué se agregó "XGBoost con balanceo de clases" como entrada separada** (no se reemplazó XGBoost normal, para poder comparar el antes/después): revisando la búsqueda de hiperparámetros, SVC/LogisticRegression/RandomForest sí exploran `class_weight="balanced"` como opción (ver `suggest_params` abajo), pero **XGBoost, GradientBoosting y KNN no tienen ningún mecanismo para compensar el desbalance de clases** (la clase 2/maligna es ~23% de los datos). Eso no es casualidad de los datos, es una asimetría real en cómo se armó la búsqueda. Además, XGBoost tiene el ROC AUC más alto de los 6 (0.88) pero la sensibilidad más baja (0.49) -- su capacidad de *ordenar* pacientes por riesgo es la mejor de todas, pero el punto de corte que elige es muy conservador por no tener manera de decirle "la clase minoritaria importa más". Balancear sus pesos de entrenamiento es la forma directa de probar si esa hipótesis es cierta.

In [ ]:
import optuna
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

from julieta.models.metrics import ClassificationMetrics

optuna.logging.set_verbosity(optuna.logging.WARNING)


class BalancedXGBClassifier(XGBClassifier):
    """XGBClassifier con `sample_weight` balanceado automático en cada `.fit()`.

    XGBoost (a diferencia de SVC/LogisticRegression/RandomForest) no tiene un
    parámetro `class_weight` nativo -- solo acepta `sample_weight` explícito
    en `.fit()`. Sobrescribiendo `.fit()` para calcular ese peso automáticamente
    a partir de la `y` que le llegue, este wrapper se comporta EXACTAMENTE
    igual que XGBClassifier en todo lo demás (mismos hiperparámetros, mismo
    predict/predict_proba), y además queda seguro para cross-validation: cada
    fold de `cross_val_predict` llama a `.fit()` con SOLO su propia porción de
    entrenamiento, así que el peso se calcula por fold (sin fuga de
    información del fold de validación).
    `compute_sample_weight("balanced", y)` le da más peso a las filas de la
    clase minoritaria (clase 2 = maligno, ~23% de los datos) durante el
    entrenamiento -- el mismo efecto que `class_weight="balanced"` en sklearn,
    pero implementado a mano porque XGBoost no lo expone directamente.
    """

    def fit(self, X, y, **kwargs):
        sample_weight = compute_sample_weight("balanced", y)
        return super().fit(X, y, sample_weight=sample_weight, **kwargs)


# Reducido tras un timeout real de 30 min con 30 trials y hasta 500 árboles
# por modelo -- GradientBoostingClassifier no se puede paralelizar en
# sklearn (a diferencia de RF/XGB), así que era el cuello de botella.
CLASSIFIERS_TO_TRY = [
    "SVC",
    "LogisticRegression",
    "RandomForestClassifier",
    "XGBClassifier",
    "XGBClassifier_balanced",
    "GradientBoostingClassifier",
    "KNeighborsClassifier",
]
N_TRIALS_PER_CLASSIFIER = 20
CV_N_JOBS = -1  # paraleliza los folds de cross_val_predict, no solo el modelo

# Misma semilla que el notebook de referencia (resistencia/reactancia), para
# que la ÚNICA diferencia entre ambos experimentos sea la base de features
# (magnitud/fase vs R/X), no el azar de la búsqueda de hiperparámetros.
OPTUNA_SEED = 369

# Experimento MLflow SEPARADO del original (entrenamiento_healthy_multiclasificador_qc)
# para no mezclar corridas de bases de features distintas en la misma tabla.
mlflow.set_experiment("entrenamiento_healthy_multiclasificador_magphase_groupsplit")


def suggest_params(trial, model_name):
    """Define el espacio de búsqueda de hiperparámetros de cada clasificador.

    "XGBClassifier_balanced" reusa exactamente el mismo espacio que
    "XGBClassifier" (mismos rangos de n_estimators/max_depth/learning_rate/
    subsample) -- la ÚNICA diferencia entre ambos es el balanceo de clases
    en build_pipeline() de abajo, para que la comparación sea justa (si
    mejora, es por el balanceo, no porque se le dio una búsqueda distinta).
    """
    if model_name == "SVC":
        # C: qué tan estricta es la frontera de decisión (log-scale porque el
        # efecto de C es multiplicativo, no aditivo). gamma: alcance de cada
        # punto de soporte en el kernel rbf. class_weight tunable porque SVC
        # sí lo soporta nativo -- se prueba con y sin balanceo.
        return dict(
            C=trial.suggest_float("C", 1e-3, 1e3, log=True),
            gamma=trial.suggest_float("gamma", 1e-4, 1e1, log=True),
            kernel=trial.suggest_categorical("kernel", ["rbf", "linear"]),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
        )
    if model_name == "LogisticRegression":
        return dict(
            C=trial.suggest_float("C", 1e-3, 1e3, log=True),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
            solver="lbfgs",
            max_iter=2000,
        )
    if model_name == "RandomForestClassifier":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 50, 200),
            max_depth=trial.suggest_int("max_depth", 2, 15),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
        )
    if model_name in ("XGBClassifier", "XGBClassifier_balanced"):
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 50, 200),
            max_depth=trial.suggest_int("max_depth", 2, 8),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
        )
    if model_name == "GradientBoostingClassifier":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 30, 100),
            max_depth=trial.suggest_int("max_depth", 2, 5),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        )
    if model_name == "KNeighborsClassifier":
        return dict(
            n_neighbors=trial.suggest_int("n_neighbors", 3, 30),
            weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
        )
    raise ValueError(model_name)


def build_pipeline(model_name, params):
    """Instancia el clasificador + escalado (MinMaxScaler, 0-1) como un Pipeline.

    Todos con random_state=369 fijo para que la única fuente de variación
    entre corridas sea el código, no el azar del propio modelo.
    """
    if model_name == "SVC":
        clf = SVC(probability=True, random_state=369, **params)
    elif model_name == "LogisticRegression":
        clf = LogisticRegression(random_state=369, **params)
    elif model_name == "RandomForestClassifier":
        clf = RandomForestClassifier(random_state=369, **params)
    elif model_name == "XGBClassifier":
        clf = XGBClassifier(random_state=369, eval_metric="logloss", **params)
    elif model_name == "XGBClassifier_balanced":
        # Única diferencia real contra "XGBClassifier": la clase, no los
        # hiperparámetros (ver BalancedXGBClassifier arriba).
        clf = BalancedXGBClassifier(random_state=369, eval_metric="logloss", **params)
    elif model_name == "GradientBoostingClassifier":
        clf = GradientBoostingClassifier(random_state=369, **params)
    elif model_name == "KNeighborsClassifier":
        clf = KNeighborsClassifier(**params)
    else:
        raise ValueError(model_name)
    return Pipeline([("scaler", MinMaxScaler()), ("clf", clf)])


results = {}
best_pipes = {}

for model_name in CLASSIFIERS_TO_TRY:
    print(f"--- {model_name} ---")

    def objective(trial, model_name=model_name):
        """Función objetivo de Optuna: entrena+valida con 5-fold CV y devuelve J de Youden.

        cross_val_predict entrena el pipeline 5 veces (una por fold, con
        `cv` definido en la celda de arriba) y junta las predicciones de
        validación de los 5 folds en un solo vector -- así ClassificationMetrics
        calcula sensibilidad/especificidad sobre TODO X_train_val, pero cada
        predicción individual vino de un modelo que nunca vio esa fila durante
        su propio entrenamiento.
        """
        params = suggest_params(trial, model_name)
        pipe = build_pipeline(model_name, params)
        y_pred = cross_val_predict(
            pipe, X_train_val, y_train_val, cv=cv, method="predict", n_jobs=CV_N_JOBS
        )
        fold_metrics = ClassificationMetrics(y_true=y_train_val, y_pred=y_pred, triclass=triclass)
        return fold_metrics.sensitivity() + fold_metrics.specificity() - 1

    study = optuna.create_study(
        direction="maximize",
        study_name=f"{model_name}_youden_j",
        sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS_PER_CLASSIFIER, show_progress_bar=False)

    # Con los mejores hiperparámetros encontrados por Optuna (sobre CV), se
    # entrena UNA VEZ MÁS con TODO X_train_val (no solo 4/5 folds) y se evalúa
    # contra X_test -- el conjunto que ningún paso anterior tocó.
    best_pipe = build_pipeline(model_name, study.best_params)
    best_pipe.fit(X_train_val, y_train_val)

    # Evaluación en TRAIN (in-sample: el mismo X_train_val con el que se
    # entrenó, NO las predicciones fuera-de-fold de cross_val_predict de
    # arriba) -- esto es deliberadamente optimista, y esa es la idea: la
    # brecha entre estas métricas de train y las de test de abajo es la
    # señal clásica de sobreajuste. Si train sale ~1.0 y test muy por debajo,
    # el modelo memorizó en vez de generalizar.
    y_pred_train = best_pipe.predict(X_train_val)
    try:
        y_proba_train = best_pipe.predict_proba(X_train_val)[:, -1]
    except Exception:
        y_proba_train = None
    train_metrics = ClassificationMetrics(
        y_true=y_train_val, y_pred=y_pred_train, y_proba=y_proba_train, triclass=triclass
    )
    train_metrics_dict = train_metrics.get_metrics()
    train_metrics_dict["youden_j"] = (
        train_metrics_dict["sensitivity"] + train_metrics_dict["specificity"] - 1
    )

    # Evaluación en TEST (la que de verdad importa para comparar modelos)
    y_pred_test = best_pipe.predict(X_test)
    try:
        y_proba_test = best_pipe.predict_proba(X_test)[:, -1]
    except Exception:
        y_proba_test = None

    test_metrics = ClassificationMetrics(
        y_true=y_test, y_pred=y_pred_test, y_proba=y_proba_test, triclass=triclass
    )
    metrics_dict = test_metrics.get_metrics()
    metrics_dict["youden_j"] = metrics_dict["sensitivity"] + metrics_dict["specificity"] - 1
    results[model_name] = metrics_dict
    best_pipes[model_name] = best_pipe

    # MLflow: un run por clasificador, con params, métricas, matriz de
    # confusión/reporte de TRAIN y de TEST por separado (como figuras), y el
    # modelo -- para poder inspeccionar y comparar esta corrida completa
    # desde la UI de MLflow, incluyendo la brecha train-test de un vistazo.
    with mlflow.start_run(run_name=model_name):
        mlflow.set_tag("dataset", "healthy_advanced_magphase_dataset_qc.csv (split por paciente)")
        mlflow.set_tag("feature_basis", "magnitude_phase")
        mlflow.set_tag("n_studies", "SURA,CAFAM,CLINICA_DE_MAMA")
        mlflow.log_param("model_family", model_name)
        mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
        mlflow.log_param("n_trials", N_TRIALS_PER_CLASSIFIER)
        mlflow.log_param("optuna_seed", OPTUNA_SEED)
        mlflow.log_param("n_features", X_train_val.shape[1])
        mlflow.log_param("n_train_val", X_train_val.shape[0])
        mlflow.log_param("n_test", X_test.shape[0])
        mlflow.log_metric("youden_j_cv", study.best_value)
        mlflow.log_metrics({f"train_{k}": v for k, v in train_metrics_dict.items()})
        mlflow.log_metrics({f"test_{k}": v for k, v in metrics_dict.items()})
        mlflow.log_figure(train_metrics.plot_confusion_matrix(), f"{model_name}_cm_train.png")
        mlflow.log_figure(train_metrics.plot_classification_report(), f"{model_name}_cr_train.png")
        mlflow.log_figure(test_metrics.plot_confusion_matrix(), f"{model_name}_cm_test.png")
        mlflow.log_figure(test_metrics.plot_classification_report(), f"{model_name}_cr_test.png")
        # mlflow.sklearn.log_model(best_pipe, name=model_name) -- roto contra
        # el tracking server de Azure ML: en mlflow>=3 siempre intenta crear una
        # entidad "Logged Model" via POST /api/2.0/mlflow/logged-models, que ese
        # server no implementa (404 real, confirmado). joblib + log_artifact
        # logra lo mismo (modelo descargable desde el run) sin ese endpoint.
        import tempfile

        with tempfile.TemporaryDirectory() as tmp_dir:
            model_path = f"{tmp_dir}/{model_name}.joblib"
            joblib.dump(best_pipe, model_path)
            mlflow.log_artifact(model_path)

    print(
        f"{model_name}: J_cv={study.best_value:.2f} | train_J={train_metrics_dict['youden_j']:.2f} | test={metrics_dict}"
    )

In [ ]:
# Tabla final: las 7 configuraciones ordenadas por J de Youden en TEST
# (nunca en cross-validation -- CV se usó solo para elegir hiperparámetros).
comparison = pd.DataFrame(results).T
comparison["youden_j"] = comparison["sensitivity"] + comparison["specificity"] - 1
comparison = comparison.sort_values("youden_j", ascending=False)
print(comparison)

# El ganador es, simplemente, el primero de la tabla ordenada.
winner_name = comparison.index[0]
winner_pipe = best_pipes[winner_name]
print(f"\nGanador por J de Youden en test: {winner_name}")
print(comparison.loc[winner_name].to_dict())

# Nombre de archivo distinto al experimento de referencia (resistencia/
# reactancia) -- NUNCA sobrescribe healthy_from_master_multiclasificador_ganador.pkl.
joblib.dump(
    {
        "model": winner_pipe,
        "approach": winner_name,
        "metrics": comparison.loc[winner_name].to_dict(),
    },
    models_dir("healthy_from_master_multiclasificador_ganador_magphase_groupsplit.pkl"),
)
print(
    f"\nModelo guardado: models/healthy_from_master_multiclasificador_ganador_magphase_groupsplit.pkl ({winner_name})"
)

# Matriz de confusión y reporte de clasificación del ganador, para
# inspección visual inmediata acá mismo (además de quedar como figura en
# MLflow para ese mismo run).
winner_test_metrics = ClassificationMetrics(
    y_true=y_test,
    y_pred=winner_pipe.predict(X_test),
    y_proba=winner_pipe.predict_proba(X_test)[:, -1]
    if hasattr(winner_pipe, "predict_proba")
    else None,
    triclass=triclass,
)
winner_test_metrics.plot_confusion_matrix(return_fig=False)
winner_test_metrics.plot_classification_report(return_fig=False)